# Swan — Data check for the questionnaire

## MVP scope decided on 23/10/2026

- **Concerne: **acne concern**
- **Output for MVP:** Moisturizers

## 📦 Data sources

| Source | File | Downloaded | Used here? |
|---|---|---|---|
| Open Beauty Facts | `raw_data/fr.openbeautyfacts.org.products.csv` | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex VI (UV filters) | `raw_data/eu_uv_filters_annex_vi.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex III (fragrance allergens) | `raw_data/eu_fragrance_allergens_annex_iii.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| Open Beauty Facts — Parquet on Hugging Face (`openfoodfacts/product-database`, `beauty.parquet`, 64.4 MB) | — | — | ❌ Not used — alternative source to discuss with Lina |
| EU glossary of common ingredient names — [Decision (EU) 2025/1175](https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=OJ:L_202501175) | — | — | ❌ No (useful for J1, INCI normalisation) |
| CosIng — EU cosmetic ingredient database | — | — | ❌ Not yet (J1, tasks T09 / T11) |


## 🎯  Questionnaire for MVP 
| # | Question | Profile field | Options shown → stored value | Required | Action | Based on |
|---|---|---|---|---|---|---|
| 1 | Are you 18 or older? | `is_adult` | yes → `true` / no → `false` | ✅ | refer if `false` | scope rule |
| 2 | What would you like help with? | `concern` | a simple routine for acne-prone skin → `acne_prone` / something else → `other` | ✅ | out-of-scope message if `other` | MVP scope  |
| 3 | How would you describe your blemishes? | `blemish_level` | rank from 1 to 5 | ✅ |  | referral rule | Validated on 26/10
| 4 | Does any of these apply to you? | `referral_flags` | prescribed treatment → `prescribed_treatment` / sudden or unusual lesion → `unusual_lesion` / pregnant or breastfeeding → `pregnant_or_breastfeeding` / none of these → empty list | ✅ | refer if not empty | referral rule |
| 5 | What is your skin type? | `skin_type` | `oily` / `combination` / `normal` / `dry` / not sure → `not_sure` | ✅ | rank | data to review |
| 6 | Does your skin react easily (stinging, redness)? | `reactivity` | `never` / `sometimes` / `often` | ✅ | rank (penalise fragrance) | data to review | GO with Yes or No for the MVP
| 7 | Ingredients you prefer to avoid? | `avoid` | fragrance → `fragrance` / essential oils → `essential_oils` / none → empty list | ❌ | filter 
| 8 | Which texture do you prefer? | `texture` | light → `light` / rich → `rich` / no preference → `no_preference` | ❌ (default `no_preference`) | NOT FOR THE MVP 


## To validate on 26/10

| # | Decision | Proposal | Decision |
|---|---|---|---|
| B02 | MVP | Adults with acne-prone skin, mild blemishes · cleanser + moisturizer + facial sunscreen · Number of products we need in each category (cleanser, moisturizer, sunscreen).| 1 issue acne - output moisturizer
| B04 | Market scope | EU (EU + EEA as an option: the regulation has "EEA relevance") · France is the core of the data | Start with France as a start
| — | Role edge cases | SPF moisturizer stays a moisturizer (sunscreen slot = dedicated sunscreens) · toners out · micellar water = cleanser · face lotions after examples · anti-ageing out | To split into to different categories moisterizer and sunscreen based on the tags.
| T12 | Questionnaire v1 + profile schema | 8 questions, safety first 
| T23 | Test-profile grid | Built from the profile fields; first 6 profiles in section 11
| T27 | Mockups | 4 screens: welcome, questionnaire, routine, referral 
| T06 | Data source | Same snapshot for the whole team (this CSV or `beauty.parquet`)

**File facts** :
- Number of lines : 64 238 
- About 200 columns, most of them food nutrition columns (`*_100g`) inherited from Open Food Facts


## Settings to run my notebook

In [70]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [92]:
import pandas as pd

INPUT_PATH = "../raw_data/moisturizers_with_cosing.csv"
COSING_PATH = "../raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv"

SCOPE_COLUMN = "france"            # France to start ("eu" to compare)
NB_ROUTINE_PRODUCT = 3             # moisturizers shown per profile
BLEMISH_REFERRAL_LEVEL = 4         # Q3: from this level → referral (to validate)
MIN_INGREDIENTS = 5                # fewer ingredients = list not usable (to validate with Lina)
MIN_PRODUCTS_PER_INGREDIENT = 5    # an ingredient becomes a column if at least 5 products contain it - just as a test, can be changed
AVOID_FRAGRANCE_INCLUDES_ALLERGENS = True   # Q7 'avoid fragrance' also removes fragrance allergens (team decision)

# Word lists for the signals — names from EU Regulation 1223/2009, Annex III (fragrance allergens)
FRAGRANCE_ALLERGENS = ["linalool", "limonene", "citronellol", "geraniol", "citral", "coumarin", "eugenol",
                       "benzyl alcohol", "benzyl benzoate", "benzyl salicylate", "hexyl cinnamal", "farnesol",
                       "alpha-isomethyl ionone", "hydroxycitronellal", "cinnamal", "amyl cinnamal", "isoeugenol"]
ESSENTIAL_OILS = ["lavandula", "citrus", "mentha", "eucalyptus", "rosmarinus", "melaleuca", "pelargonium",
                  "cymbopogon", "cananga", "santalum", "pogostemon", "juniperus", "cinnamomum", "jasminum"]
ACNE_ACTIVES = ["salicylic acid", "niacinamide", "zinc pca", "zinc gluconate", "azelaic acid"]

## Load Data Set and check the columns

In [93]:
moist = pd.read_csv(INPUT_PATH, dtype={"code": str})
print(moist.shape)
print("France:", (moist["france"] == True).sum(), "| EU:", (moist["eu"] == True).sum())
moist[["code", "product_name", "brands", "ingredients_text"]].head()

(244, 17)
France: 114 | EU: 170


,code,product_name,brands,ingredients_text
0,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,Water. Glycerin. Copper Tripeptide-1. Phenoxye...
1,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,Water. Glycerin. Copper Tripeptide-1. Phenoxye...
2,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,"AQUA, NIACINAMIDE, DIBUTYL ADIPATE, GLYCERIN, ..."
3,8437018443420,RF Facial Cream,Masderm,"Aqua, Paraffinum Liquidum, Glyceryl Stearate, ..."
4,7891010255725,Hydro Boost,Neutrogena,"Aqua/água purificada, Dimethicone/dimeticona, ..."


## Check product type

In [94]:
def wrong_type(name):
    """Return the wrong product type found in the name ('' = looks like a face moisturizer)."""
    text = str(name).lower()
    if "nettoy" in text or "cleans" in text or "cleaning" in text or "foaming" in text:
        return "cleanser"
    elif "tonic" in text or "tonique" in text or "toner" in text:
        return "toner"
    elif "hand" in text or "mains" in text:
        return "hand"
    elif "yeux" in text or "eye" in text or "contour" in text:
        return "eye"
    elif text.startswith("huile") or " oil for" in text:
        return "oil"
    elif "mask" in text or "masque" in text or "mascarilla" in text:
        return "mask"
    elif "stick" in text or "gommage" in text or "désincrustant" in text:
        return "other type"
    else:
        return ""

moist["exclusion_reason"] = moist["product_name"].apply(wrong_type)
moist[moist["exclusion_reason"] != ""][["product_name", "exclusion_reason"]]

,product_name,exclusion_reason
0,PLU Laboratories GHK-Cu Face Tonic,toner
1,PLU Laboratories GHK-Cu Face Tonic,toner
34,Mains à Croquer,hand
103,Rice Bran Oil for Face,oil
125,Mascarilla hidratante nocturna,mask
127,Crème Nettoyante Désincrustante anti-points noirs,cleanser
144,Effaclar Deep Cleaning Foaming Crem,cleanser
146,huile de jojoba vierge,oil
147,huile de chanvre bio,oil
216,Soin anti-âge yeux rechargeur jeunesse et perf...,eye


In [95]:
#Check the exclusion based on product name for MVP
names = moist["product_name"].str.lower().fillna("")
moist["is_anti_ageing"] = names.str.contains("anti-âge|anti-age|rides|wrinkle|firming|lift")
moist["is_baby"] = names.str.contains("bébé|bebe|baby")
moist["is_face_and_body"] = names.str.contains("corps|body")
moist[["is_anti_ageing", "is_baby", "is_face_and_body"]].sum()

is_anti_ageing      36
is_baby              2
is_face_and_body     7
dtype: int64

In [96]:
#Check duplicates
moist["name_brand"] = names + " | " + moist["brands"].str.lower().fillna("")
moist["is_duplicate"] = moist["name_brand"].duplicated()
moist.loc[(moist["exclusion_reason"] == "") & moist["is_duplicate"], "exclusion_reason"] = "duplicate"
print("Duplicates:", moist["is_duplicate"].sum())

Duplicates: 7


In [ ]:
#Products whose name doesn't say what they are (e.g. "Sébium sensitive") → assumption by looking at the name
product_words = "crème|creme|cream|crema|krem|moistur|hydrat|gel|fluid|baume|balm|soin|lotion|emulsion"
moist["to_check_by_hand"] = ~names.str.contains(product_words)
print("To check by hand:", moist["to_check_by_hand"].sum())
moist[moist["to_check_by_hand"]][["code", "product_name", "brands"]]

1518 INCI names in the reference


## One Name per ingredient

In [97]:
cosing = pd.read_csv(COSING_PATH, dtype=str)
cosing["inci"] = cosing["inci_name"].str.lower()
cosing = cosing.drop_duplicates("inci")      # 3 names appear twice in the reference
inci_names = list(cosing["inci"])
print(len(inci_names), "INCI names in the reference")

1518 INCI names in the reference


In [98]:
#Split the ingredient into a list
def split_ingredients(text):
    """Turn the ingredient text into a list of names."""
    text = str(text).lower()
    text = text.replace("ingredients:", "").replace("ingrédients:", "")
    text = text.replace("•", ",")                                # some labels use • instead of ,
    text = text.replace("1,2-hexanediol", "1-2-hexanediol")     # this comma is part of the name
    if "," not in text:
        text = text.replace(". ", ",")                          # some labels use '. ' instead of ','
    names = []
    for piece in text.split(","):
        name = piece.strip().strip(".").strip()
        if name != "":
            names.append(name.replace("1-2-hexanediol", "1,2-hexanediol"))
    return names

ALIASES = {
    "water": "aqua", "eau": "aqua",
    "fragrance": "parfum", "perfume": "parfum",
    "glycerine": "glycerin",
    "shea butter": "butyrospermum parkii butter", "butyrospermum parkii": "butyrospermum parkii butter",
    "alcohol denat": "alcohol denat.",
}

def to_inci(name):
    """Clean one ingredient name and return its INCI name when we know it."""
    name = name.lower().replace("*", "").strip().strip(".").strip()
    if "(" in name and ")" in name:                      # 'parkii (shea) butter' -> 'parkii butter'
        name = name.split("(")[0] + name.split(")")[-1]
    elif "(" in name:                                    # 'aqua (water' (bracket not closed) -> 'aqua'
        name = name.split("(")[0]
    name = " ".join(name.split())
    name = ALIASES.get(name, name)
    if name not in inci_names and "/" in name:           # 'aqua/water/eau' -> 'aqua'
        first = ALIASES.get(name.split("/")[0].strip(), name.split("/")[0].strip())
        if first in inci_names:
            name = first
    return name

print(split_ingredients("Water. Glycerin. Phenoxyethanol."))
print(to_inci("Aqua / Water"), "|", to_inci("butyrospermum parkii (shea) butter*"), "|", to_inci("parfum (fragrance)."))

['water', 'glycerin', 'phenoxyethanol']
aqua | butyrospermum parkii butter | parfum


## Generate the table with one raw - one product x one ingredient 

In [99]:
rows = []
for code, text in zip(moist["code"], moist["ingredients_text"]):
    position = 1
    for raw_name in split_ingredients(text):
        rows.append({"code": code, "position": position, "raw_name": raw_name, "inci": to_inci(raw_name)})
        position = position + 1

long = pd.DataFrame(rows)
long = long[long["inci"] != ""]
long["in_cosing"] = long["inci"].isin(inci_names)

print("Rows:", len(long))
print("Different names before / after:", long["raw_name"].nunique(), "/", long["inci"].nunique())
print("Found in CosIng:", round(long["in_cosing"].mean() * 100, 1), "%   (team matching: 84.8 %)")

Rows: 6472
Different names before / after: 1747 / 1505
Found in CosIng: 91.6 %   (team matching: 84.8 %)


In [ ]:
long[~long["in_cosing"]]["inci"].value_counts().head(20)

inci
palmitoyl oligopeptide                            5
_prunus amygdalus_ dulcis oil                     3
brassica juncea seed oil                          2
lysine hci                                        2
coco caprylate                                    2
ingredients/ aqua                                 2
ginkgo bilboa leaf extract                        2
caprylic                                          2
cl 16035                                          2
aus kontrolliert biologischem anbau               2
caprylic triglyceride                             2
capric triglyceride                               2
helianthus annus seed oil                         2
acrylates / c10-30 alkyl acrylate crosspolymer    2
avene thermal spring water                        2
glycine soja                                      2
cl 77891                                          2
alpha-isomethyl lonone                            2
butylphenyl methylproponal                        2
lonicer

In [100]:
moist["nb_ingredients"] = moist["code"].map(long.groupby("code")["inci"].count()).fillna(0)
moist["coverage"] = moist["code"].map(long.groupby("code")["in_cosing"].mean()).fillna(0)
moist.loc[(moist["exclusion_reason"] == "") & (moist["nb_ingredients"] < MIN_INGREDIENTS), "exclusion_reason"] = "list too short"

moist["in_scope"] = (moist["exclusion_reason"] == "") & (moist[SCOPE_COLUMN] == True)
print(moist["exclusion_reason"].value_counts())
print("In scope (" + SCOPE_COLUMN + "):", moist["in_scope"].sum(), "| same rules in the EU:", ((moist["exclusion_reason"] == "") & (moist["eu"] == True)).sum())

exclusion_reason
                  213
list too short     11
duplicate           6
oil                 4
cleanser            3
toner               2
other type          2
hand                1
mask                1
eye                 1
Name: count, dtype: int64
In scope (france): 96 | same rules in the EU: 147


## TRY to standardize based on the questionnaire 

In [101]:
moist["inci_text"] = moist["code"].map(long.groupby("code")["inci"].apply(" | ".join)).fillna("")
moist["top5_text"] = moist["code"].map(long[long["position"] <= 5].groupby("code")["inci"].apply(" | ".join)).fillna("")

moist["has_parfum"] = moist["inci_text"].str.contains("parfum")
moist["has_fragrance_allergen"] = moist["inci_text"].str.contains("|".join(FRAGRANCE_ALLERGENS))
moist["has_essential_oil"] = moist["inci_text"].str.contains("|".join(ESSENTIAL_OILS))
moist["has_acne_active"] = moist["inci_text"].str.contains("|".join(ACNE_ACTIVES))
moist["is_rich"] = moist["top5_text"].str.contains("butter|oil|cera|petrolatum|paraffinum") | names.str.contains("riche|rich|nourri|baume")
moist["is_light"] = names.str.contains("gel|fluid|fluide|light|légère")

signals = ["has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_rich", "is_light",
           "is_anti_ageing", "is_baby", "is_face_and_body"]
(moist[moist["in_scope"]][signals].mean() * 100).round(1)

has_parfum                80.2
has_fragrance_allergen    53.1
has_essential_oil         10.4
has_acne_active           10.4
is_rich                   54.2
is_light                   5.2
is_anti_ageing            24.0
is_baby                    2.1
is_face_and_body           6.2
dtype: float64

## Put ingredients as a column 

In [102]:
count = long.groupby("inci")["code"].nunique()
frequent = list(count[count >= MIN_PRODUCTS_PER_INGREDIENT].index)
print("Different ingredients:", len(count), "→ columns kept:", len(frequent))

frequent_rows = long[long["inci"].isin(frequent)]
matrix = pd.crosstab(frequent_rows["code"], frequent_rows["inci"])
matrix = (matrix > 0).astype(int)          # 1 = the product contains it, 0 = it doesn't
print(matrix.shape)

Different ingredients: 1505 → columns kept: 252
(235, 252)


In [116]:
#Products whose name doesn't say what they are (e.g. "Sébium sensitive") → for Lina to check by hand
names = moist["product_name"].str.lower().fillna("")
product_words = "crème|creme|cream|crema|krem|moistur|hydrat|gel|fluid|baume|balm|soin|lotion|emulsion"
moist["to_check_by_hand"] = ~names.str.contains(product_words)
print("To check by hand:", moist["to_check_by_hand"].sum())

To check by hand: 62


In [119]:
columns = ["code", "product_name", "brands", "france", "eu", "in_scope", "exclusion_reason", "is_duplicate",
           "to_check_by_hand", "nb_ingredients", "coverage", "inci_text"] + signals
moist[columns].to_csv("../raw_data/draft_for_lina_moisturizers.csv", index=False)
long.to_csv("../raw_data/draft_for_lina_ingredients_long.csv", index=False)
matrix.to_csv("../raw_data/draft_for_lina_ingredient_matrix.csv")
print("3 draft files saved")

3 draft files saved


## The questionnaire 

In [104]:
QUESTIONNAIRE = {
    "is_adult": [True, False],
    "concern": ["acne_prone", "other"],
    "blemish_level": [1, 2, 3, 4, 5],
    "referral_flags": [[], ["prescribed_treatment"], ["unusual_lesion"], ["pregnant_or_breastfeeding"]],
    "skin_type": ["oily", "combination", "normal", "dry", "not_sure"],
    "reactivity": [True, False],
    "avoid": [[], ["fragrance"], ["essential_oils"], ["fragrance", "essential_oils"]],
}

def check_profile(profile):
    """List the problems of a profile (empty list = valid)."""
    problems = []
    for field in QUESTIONNAIRE:
        if field not in profile:
            problems.append("missing " + field)
        elif profile[field] not in QUESTIONNAIRE[field]:
            problems.append("wrong value for " + field)
    return problems

print(check_profile({"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                     "skin_type": "oily", "reactivity": True, "avoid": []}))
print(check_profile({"is_adult": True, "concern": "acne", "skin_type": "very oily"}))

[]
['wrong value for concern', 'missing blemish_level', 'missing referral_flags', 'wrong value for skin_type', 'missing reactivity', 'missing avoid']


**Data contract** — what each answer needs from the catalogue (for Lina, Loubna, Imane):

| Question | Profile field | Catalogue column(s) |
|---|---|---|
| Q1 age · Q2 concern · Q3 blemishes · Q4 special cases | `is_adult`, `concern`, `blemish_level`, `referral_flags` | none (referral rule) |
| Q5 skin type | `skin_type` | `is_rich`, `is_light` |
| Q6 reactivity | `reactivity` | `has_parfum`, `has_essential_oil` |
| Q7 avoid | `avoid` | `has_parfum`, `has_fragrance_allergen`, `has_essential_oil` |
| Concern (acne-prone) | — | `has_acne_active`, `is_anti_ageing` |

In [106]:
contract = ["is_rich", "is_light", "has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_anti_ageing"]
print("Columns missing in the catalogue:", [column for column in contract if column not in moist.columns])

Columns missing in the catalogue: []


In [107]:
def needs_referral(profile):
    """True if we must refer the person instead of recommending products."""
    if profile["is_adult"] == False:
        return True
    elif profile["concern"] != "acne_prone":
        return True
    elif profile["blemish_level"] >= BLEMISH_REFERRAL_LEVEL:
        return True
    elif len(profile["referral_flags"]) > 0:
        return True
    else:
        return False

def eligible_products(profile, catalogue):
    """In-scope products left after the 'avoid' answers."""
    selected = catalogue[catalogue["in_scope"]]
    if "fragrance" in profile["avoid"]:
        selected = selected[~selected["has_parfum"]]
        if AVOID_FRAGRANCE_INCLUDES_ALLERGENS:
            selected = selected[~selected["has_fragrance_allergen"]]
    if "essential_oils" in profile["avoid"]:
        selected = selected[~selected["has_essential_oil"]]
    return selected

def simple_score(profile, selected):
    """PLACEHOLDER score — Loubna's score will replace it (T15). Each point has a reason."""
    score = selected["has_acne_active"].astype(int) - selected["is_anti_ageing"].astype(int)
    if profile["reactivity"] == True:
        score = score - selected["has_parfum"].astype(int) - selected["has_essential_oil"].astype(int)
    if profile["skin_type"] in ["oily", "combination"]:
        score = score + selected["is_light"].astype(int) - selected["is_rich"].astype(int)
    elif profile["skin_type"] == "dry":
        score = score + selected["is_rich"].astype(int)
    return score

def recommend(profile, catalogue):
    """Answers -> 'referral', 'not enough products', or the top moisturizers."""
    if needs_referral(profile):
        return "referral"
    selected = eligible_products(profile, catalogue).copy()
    if len(selected) < NB_ROUTINE_PRODUCT:
        return "not enough products"
    selected["score"] = simple_score(profile, selected)
    return selected.sort_values(["score", "coverage"], ascending=False).head(NB_ROUTINE_PRODUCT)

## Test some testing profiles

In [108]:
AVOID_OPTIONS = QUESTIONNAIRE["avoid"]
profiles = []
number = 1
for skin_type in QUESTIONNAIRE["skin_type"]:
    for reactivity in [True, False]:
        for blemish_level in [1, 3]:
            profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "recommendation",
                             "is_adult": True, "concern": "acne_prone", "blemish_level": blemish_level,
                             "referral_flags": [], "skin_type": skin_type, "reactivity": reactivity,
                             "avoid": AVOID_OPTIONS[number % 4], "expected": "products"})
            number = number + 1

edge_cases = [
    ("under 18",             {"is_adult": False}),
    ("other concern",        {"concern": "other"}),
    ("blemishes = 4",        {"blemish_level": 4}),
    ("blemishes = 5",        {"blemish_level": 5}),
    ("prescribed treatment", {"referral_flags": ["prescribed_treatment"]}),
    ("unusual lesion",       {"referral_flags": ["unusual_lesion"]}),
    ("pregnant",             {"referral_flags": ["pregnant_or_breastfeeding"]}),
]
for name, change in edge_cases:
    profile = {"profile_id": "P" + str(number).zfill(2), "kind": "edge case: " + name,
               "is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
               "skin_type": "oily", "reactivity": False, "avoid": [], "expected": "referral"}
    profile.update(change)
    profiles.append(profile)
    number = number + 1

profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "edge case: blemishes = 3 (just below)",
                 "is_adult": True, "concern": "acne_prone", "blemish_level": 3, "referral_flags": [],
                 "skin_type": "combination", "reactivity": True, "avoid": [], "expected": "products"})

profiles_table = pd.DataFrame(profiles)
print("Profiles:", len(profiles_table))
print(profiles_table["expected"].value_counts())
profiles_table

Profiles: 28
expected
products    21
referral     7
Name: count, dtype: int64


,profile_id,kind,is_adult,concern,blemish_level,referral_flags,skin_type,reactivity,avoid,expected
0,P01,recommendation,True,acne_prone,1,[],oily,True,[fragrance],products
1,P02,recommendation,True,acne_prone,3,[],oily,True,[essential_oils],products
2,P03,recommendation,True,acne_prone,1,[],oily,False,"[fragrance, essential_oils]",products
3,P04,recommendation,True,acne_prone,3,[],oily,False,[],products
4,P05,recommendation,True,acne_prone,1,[],combination,True,[fragrance],products
5,P06,recommendation,True,acne_prone,3,[],combination,True,[essential_oils],products
6,P07,recommendation,True,acne_prone,1,[],combination,False,"[fragrance, essential_oils]",products
7,P08,recommendation,True,acne_prone,3,[],combination,False,[],products
8,P09,recommendation,True,acne_prone,1,[],normal,True,[fragrance],products
9,P10,recommendation,True,acne_prone,3,[],normal,True,[essential_oils],products


In [109]:
for profile in profiles:
    answers = {field: profile[field] for field in QUESTIONNAIRE}
    assert check_profile(answers) == [], profile["profile_id"]

for field in ["skin_type", "reactivity", "blemish_level"]:
    print(field, ":", sorted(profiles_table[field].astype(str).unique()))
print("avoid :", profiles_table["avoid"].astype(str).unique())

skin_type : ['combination', 'dry', 'normal', 'not_sure', 'oily']
reactivity : ['False', 'True']
blemish_level : ['1', '2', '3', '4', '5']
avoid : ["['fragrance']" "['essential_oils']" "['fragrance', 'essential_oils']"
 '[]']


In [90]:
to_save = profiles_table.copy()
to_save["avoid"] = to_save["avoid"].apply("+".join)                     # ['fragrance', 'essential_oils'] -> 'fragrance+essential_oils'
to_save["referral_flags"] = to_save["referral_flags"].apply("+".join)   # [] -> '' (empty)
to_save.to_csv("../raw_data/swan_test_profiles_v1.csv", index=False)
print("Saved: swan_test_profiles_v1.csv")

Saved: swan_test_profiles_v1.csv


In [110]:
to_save = profiles_table.copy()
to_save["avoid"] = to_save["avoid"].apply("+".join)                     # ['fragrance', 'essential_oils'] -> 'fragrance+essential_oils'
to_save["referral_flags"] = to_save["referral_flags"].apply("+".join)   # [] -> '' (empty)
to_save.to_csv("../raw_data/swan_test_profiles_v1.csv", index=False)
print("Saved: swan_test_profiles_v1.csv")

Saved: swan_test_profiles_v1.csv


In [111]:
result_rows = []
for profile in profiles:
    result = recommend(profile, moist)
    if type(result) == str:
        result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": result})
    else:
        rank = 1
        for _, product in result.iterrows():           # one loop turn per recommended product (row)
            respects_avoid = True
            if "fragrance" in profile["avoid"] and product["has_parfum"]:
                respects_avoid = False
            if "essential_oils" in profile["avoid"] and product["has_essential_oil"]:
                respects_avoid = False
            result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": "products",
                                "rank": rank, "code": product["code"], "product_name": product["product_name"],
                                "score": product["score"], "R2_respects_avoid": respects_avoid,
                                "R4_face_moisturizer": "", "R5_texture_fits": "", "comment": ""})
            rank = rank + 1

results = pd.DataFrame(result_rows)
results.head(10)

,profile_id,expected,outcome,rank,code,product_name,score,R2_respects_avoid,R4_face_moisturizer,R5_texture_fits,comment
0,P01,products,products,1.0,3605970360757,Ultra Facial Cream,1.0,True,,,
1,P01,products,products,2.0,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,0.0,True,,,
2,P01,products,products,3.0,3701129804445,Créaline Défensive,0.0,True,,,
3,P02,products,products,1.0,3380813064107,Gel Revitalisant,1.0,True,,,
4,P02,products,products,2.0,3605970360757,Ultra Facial Cream,1.0,True,,,
5,P02,products,products,3.0,5017634247843,Anti-boutons,0.0,True,,,
6,P03,products,products,1.0,3605970360757,Ultra Facial Cream,1.0,True,,,
7,P03,products,products,2.0,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,0.0,True,,,
8,P03,products,products,3.0,3701129804445,Créaline Défensive,0.0,True,,,
9,P04,products,products,1.0,3380813064107,Gel Revitalisant,2.0,True,,,


In [113]:
outcome_per_profile = results.groupby("profile_id")[["expected", "outcome"]].first()
print("R1 expected outcome     :", (outcome_per_profile["expected"] == outcome_per_profile["outcome"]).sum(), "/", len(outcome_per_profile))

products = results[results["outcome"] == "products"]
print("R2 respects 'avoid'     :", products["R2_respects_avoid"].sum(), "/", len(products))
print("R3 no duplicate in a top:", (products.groupby("profile_id")["code"].nunique() == NB_ROUTINE_PRODUCT).sum(), "/", products["profile_id"].nunique())

print()
print("Profiles with a wrong outcome:")
outcome_per_profile[outcome_per_profile["expected"] != outcome_per_profile["outcome"]]

R1 expected outcome     : 28 / 28
R2 respects 'avoid'     : 63 / 63
R3 no duplicate in a top: 21 / 21

Profiles with a wrong outcome:


,expected,outcome
profile_id,,


In [114]:
results.to_csv("../raw_data/swan_test_results_v1.csv", index=False)
print("Saved: swan_test_results_v1.csv —", len(products), "products to judge")

Saved: swan_test_results_v1.csv — 63 products to judge
